# Lab 02: Advertising Sales Regression — Data Preparation, One-Sample SGD, Full-Batch GD & Bias-Augmented Vectors

[Back to Session Overview](../README.md) · [Module 02 Index](../../README.md)

---

## Overview

This notebook solves **Problems 1 through 4** from the exercise curriculum using `../data/advertising.csv` ($N = 200$ markets, three advertising channels `TV`, `Radio`, `Newspaper`, and target `Sales`):

1. **Problem 1 (`prepare_data` & `get_column`):** Parse CSV rows cleanly into column-major feature lists `X = [tv, radio, newspaper]` and target list `y`.
2. **Problem 2 (One-Sample SGD):** Implement scalar `predict`, `compute_loss_mse`, `compute_loss_mae`, `compute_gradient_wi`, `compute_gradient_b`, and per-sample updates for both MSE and MAE objectives, then run inference on `(TV=19.2, Radio=35.9, Newspaper=51.3)`.
3. **Problem 3 (Full-Batch $N$-Sample GD):** Accumulate gradients across all $N$ samples before updating weights once per epoch (fixing the post-loop averaging bug from the reference snippet) and compare smooth MSE vs. MAE epoch curves.
4. **Problem 4 (Bias-Augmented Feature List):** Represent each sample as `[1, x1, x2, x3]` and unify `[b, w1, w2, w3]` into a single weight vector, verifying numerical equivalence with Problem 2.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DATA_PATH = Path("..") / "data" / "advertising.csv"
assert DATA_PATH.is_file(), f"Dataset not found at {DATA_PATH}"

plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## Problem 1: Reading & Organizing `advertising.csv`

We implement `get_column(data, index)` using a list comprehension and build `prepare_data(file_name_dataset)` returning `(X, y)` where `X = [tv_data, radio_data, newspaper_data]` has shape $3 \times 200$ and `y = sales_data` has length $200$.


In [ ]:
def get_column(data: list[list[float]], index: int) -> list[float]:
    """Extract a single column from a 2D list of rows."""
    return [row[index] for row in data]


def prepare_data(file_name_dataset: str | Path) -> tuple[list[list[float]], list[float]]:
    """Load advertising CSV into column-major features [TV, Radio, Newspaper] and Sales."""
    data = np.genfromtxt(file_name_dataset, delimiter=",", skip_header=1).tolist()

    tv_data = get_column(data, 0)
    radio_data = get_column(data, 1)
    newspaper_data = get_column(data, 2)
    sales_data = get_column(data, 3)

    x_cols = [tv_data, radio_data, newspaper_data]
    return x_cols, sales_data


X_cols, y_sales = prepare_data(DATA_PATH)
print(f"Number of feature channels: {len(X_cols)}, samples per channel: {len(X_cols[0])}")
print("First 5 TV budgets:", X_cols[0][:5])
print("First 5 Sales     :", y_sales[:5])
assert len(X_cols) == 3 and len(X_cols[0]) == 200 and len(y_sales) == 200

---

## Problem 2: One-Sample Linear Regression (Stochastic Gradient Descent)

For each sample $(x_1, x_2, x_3, y)$, the model computes:

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + b$$

We evaluate both **MSE** ($L = (\hat{y} - y)^2$, gradients $2 x_i (\hat{y} - y)$ and $2(\hat{y} - y)$) and **MAE** ($L = |\hat{y} - y|$, subgradients $x_i \,\mathrm{sign}(\hat{y} - y)$ and $\mathrm{sign}(\hat{y} - y)$) starting from the fixed benchmark parameters.


In [ ]:
def initialize_params() -> tuple[float, float, float, float]:
    """Return deterministic benchmark initial parameters from the exercise."""
    w1 = 0.016992259082509283
    w2 = 0.0070783670518262355
    w3 = -0.002307860847821344
    b = 0.0
    return w1, w2, w3, b


def predict(x1: float, x2: float, x3: float, w1: float, w2: float, w3: float, b: float) -> float:
    return w1 * x1 + w2 * x2 + w3 * x3 + b


def compute_loss_mse(y_hat: float, y: float) -> float:
    return (y_hat - y) ** 2


def compute_loss_mae(y_hat: float, y: float) -> float:
    return abs(y_hat - y)


def compute_gradient_wi(xi: float, y: float, y_hat: float) -> float:
    return 2.0 * xi * (y_hat - y)


def compute_gradient_b(y: float, y_hat: float) -> float:
    return 2.0 * (y_hat - y)


def compute_gradient_mae_wi(xi: float, y: float, y_hat: float) -> float:
    if y_hat > y:
        return xi
    if y_hat < y:
        return -xi
    return 0.0


def compute_gradient_mae_b(y: float, y_hat: float) -> float:
    if y_hat > y:
        return 1.0
    if y_hat < y:
        return -1.0
    return 0.0


def update_weight_wi(wi: float, dl_dwi: float, lr: float) -> float:
    return wi - lr * dl_dwi


def update_weight_b(b: float, dl_db: float, lr: float) -> float:
    return b - lr * dl_db


def implement_linear_regression(
    x_data: list[list[float]],
    y_data: list[float],
    epoch_max: int = 50,
    lr: float = 1e-5,
    loss_type: str = "mse",
) -> tuple[float, float, float, float, list[float]]:
    """Train linear regression with 1-sample SGD using MSE or MAE loss."""
    losses: list[float] = []
    w1, w2, w3, b = initialize_params()
    n = len(y_data)

    for _epoch in range(epoch_max):
        for i in range(n):
            x1 = x_data[0][i]
            x2 = x_data[1][i]
            x3 = x_data[2][i]
            y = y_data[i]

            y_hat = predict(x1, x2, x3, w1, w2, w3, b)

            if loss_type == "mse":
                loss = compute_loss_mse(y_hat, y)
                dl_dw1 = compute_gradient_wi(x1, y, y_hat)
                dl_dw2 = compute_gradient_wi(x2, y, y_hat)
                dl_dw3 = compute_gradient_wi(x3, y, y_hat)
                dl_db = compute_gradient_b(y, y_hat)
            elif loss_type == "mae":
                loss = compute_loss_mae(y_hat, y)
                dl_dw1 = compute_gradient_mae_wi(x1, y, y_hat)
                dl_dw2 = compute_gradient_mae_wi(x2, y, y_hat)
                dl_dw3 = compute_gradient_mae_wi(x3, y, y_hat)
                dl_db = compute_gradient_mae_b(y, y_hat)
            else:
                raise ValueError(f"Unsupported loss_type: {loss_type}")

            w1 = update_weight_wi(w1, dl_dw1, lr)
            w2 = update_weight_wi(w2, dl_dw2, lr)
            w3 = update_weight_wi(w3, dl_dw3, lr)
            b = update_weight_b(b, dl_db, lr)

            losses.append(loss)

    return w1, w2, w3, b, losses


w1_mse, w2_mse, w3_mse, b_mse, losses_sgd_mse = implement_linear_regression(
    X_cols, y_sales, epoch_max=50, lr=1e-5, loss_type="mse"
)
w1_mae, w2_mae, w3_mae, b_mae, losses_sgd_mae = implement_linear_regression(
    X_cols, y_sales, epoch_max=50, lr=1e-5, loss_type="mae"
)

# Inference check on unseen sample: TV=19.2, Radio=35.9, Newspaper=51.3
sales_pred_mse = predict(19.2, 35.9, 51.3, w1_mse, w2_mse, w3_mse, b_mse)
sales_pred_mae = predict(19.2, 35.9, 51.3, w1_mae, w2_mae, w3_mae, b_mae)
print(f"SGD-MSE params: w=({w1_mse:.5f}, {w2_mse:.5f}, {w3_mse:.5f}), b={b_mse:.5f}")
print(f"Predicted sales (MSE) for (19.2, 35.9, 51.3): {sales_pred_mse:.4f}")
print(f"SGD-MAE params: w=({w1_mae:.5f}, {w2_mae:.5f}, {w3_mae:.5f}), b={b_mae:.5f}")
print(f"Predicted sales (MAE) for (19.2, 35.9, 51.3): {sales_pred_mae:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses_sgd_mse[:100], color="#2563eb")
axes[0].set_xlabel("#iteration")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("1-Sample SGD: First 100 Iterations (MSE)")

axes[1].plot(losses_sgd_mae[:100], color="#ea580c")
axes[1].set_xlabel("#iteration")
axes[1].set_ylabel("MAE Loss")
axes[1].set_title("1-Sample SGD: First 100 Iterations (MAE)")

plt.tight_layout()
plt.show()

---

## Problem 3: Full-Batch $N$-Sample Linear Regression

Instead of updating parameters after every individual sample, full-batch gradient descent accumulates both the loss and gradients across all $N = 200$ samples during each epoch, then performs a single averaged update:

$$w_j \leftarrow w_j - \eta \cdot \frac{1}{N} \sum_{i=1}^{N} \frac{\partial \ell^{(i)}}{\partial w_j}, \qquad b \leftarrow b - \eta \cdot \frac{1}{N} \sum_{i=1}^{N} \frac{\partial \ell^{(i)}}{\partial b}$$

*(Note: this fixes the reference slide typo where `dl_dw1 / N` was written instead of the accumulated `dw1_total / N`.)*


In [ ]:
def implement_linear_regression_nsamples(
    x_data: list[list[float]],
    y_data: list[float],
    epoch_max: int = 1000,
    lr: float = 1e-5,
    loss_type: str = "mse",
) -> tuple[float, float, float, float, list[float]]:
    """Train linear regression with full-batch N-sample gradient descent."""
    losses: list[float] = []
    w1, w2, w3, b = initialize_params()
    n = len(y_data)

    for _epoch in range(epoch_max):
        loss_total = 0.0
        dw1_total = 0.0
        dw2_total = 0.0
        dw3_total = 0.0
        db_total = 0.0

        for i in range(n):
            x1 = x_data[0][i]
            x2 = x_data[1][i]
            x3 = x_data[2][i]
            y = y_data[i]

            y_hat = predict(x1, x2, x3, w1, w2, w3, b)

            if loss_type == "mse":
                loss = compute_loss_mse(y_hat, y)
                dl_dw1 = compute_gradient_wi(x1, y, y_hat)
                dl_dw2 = compute_gradient_wi(x2, y, y_hat)
                dl_dw3 = compute_gradient_wi(x3, y, y_hat)
                dl_db = compute_gradient_b(y, y_hat)
            elif loss_type == "mae":
                loss = compute_loss_mae(y_hat, y)
                dl_dw1 = compute_gradient_mae_wi(x1, y, y_hat)
                dl_dw2 = compute_gradient_mae_wi(x2, y, y_hat)
                dl_dw3 = compute_gradient_mae_wi(x3, y, y_hat)
                dl_db = compute_gradient_mae_b(y, y_hat)
            else:
                raise ValueError(f"Unsupported loss_type: {loss_type}")

            loss_total += loss
            dw1_total += dl_dw1
            dw2_total += dl_dw2
            dw3_total += dl_dw3
            db_total += dl_db

        # Averaged full-batch parameter update
        w1 = update_weight_wi(w1, dw1_total / n, lr)
        w2 = update_weight_wi(w2, dw2_total / n, lr)
        w3 = update_weight_wi(w3, dw3_total / n, lr)
        b = update_weight_b(b, db_total / n, lr)

        losses.append(loss_total / n)

    return w1, w2, w3, b, losses


bw1_mse, bw2_mse, bw3_mse, bb_mse, batch_losses_mse = implement_linear_regression_nsamples(
    X_cols, y_sales, epoch_max=1000, lr=1e-5, loss_type="mse"
)
bw1_mae, bw2_mae, bw3_mae, bb_mae, batch_losses_mae = implement_linear_regression_nsamples(
    X_cols, y_sales, epoch_max=1000, lr=1e-4, loss_type="mae"
)

print(f"Batch MSE: initial={batch_losses_mse[0]:.4f} -> final={batch_losses_mse[-1]:.4f}")
print(f"Batch MAE: initial={batch_losses_mae[0]:.4f} -> final={batch_losses_mae[-1]:.4f}")
assert batch_losses_mse[-1] < batch_losses_mse[0]
assert batch_losses_mae[-1] < batch_losses_mae[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(batch_losses_mse, color="#16a34a")
axes[0].set_xlabel("#epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Full-Batch GD (N=200): MSE Loss")

axes[1].plot(batch_losses_mae, color="#7c3aed")
axes[1].set_xlabel("#epoch")
axes[1].set_ylabel("MAE Loss")
axes[1].set_title("Full-Batch GD (N=200): MAE Loss")

plt.tight_layout()
plt.show()

---

## Problem 4: Bias-Augmented Feature List (`[1, x1, x2, x3]`)

By prepending $x_0 = 1$ to every sample row and packing weights into `[b, w1, w2, w3]`, prediction, gradient computation, and updates generalize to any feature count $d$:

- `X[i] = [1, x1, x2, x3]`
- `predict(x_features, weights) = sum(f * w for f, w in zip(x_features, weights))`
- `compute_gradient_w(x_features, y, y_hat) = [2 * xi * (y_hat - y) for xi in x_features]`
- `update_weight(weights, dl_dweights, lr) = [w - lr * dw for w, dw in zip(weights, dl_dweights)]`


In [ ]:
def prepare_data_augmented(
    file_name_dataset: str | Path,
) -> tuple[list[list[float]], list[float]]:
    """Load advertising CSV into row-major bias-augmented vectors [1, TV, Radio, Newspaper]."""
    data = np.genfromtxt(file_name_dataset, delimiter=",", skip_header=1).tolist()

    tv_data = get_column(data, 0)
    radio_data = get_column(data, 1)
    newspaper_data = get_column(data, 2)
    sales_data = get_column(data, 3)

    x_rows = [
        [1.0, x1, x2, x3] for x1, x2, x3 in zip(tv_data, radio_data, newspaper_data, strict=True)
    ]
    return x_rows, sales_data


def initialize_params_vector() -> list[float]:
    w1, w2, w3, b = initialize_params()
    return [b, w1, w2, w3]


def predict_vec(x_features: list[float], weights: list[float]) -> float:
    return sum(f * w for f, w in zip(x_features, weights, strict=True))


def compute_gradient_w(x_features: list[float], y: float, y_hat: float) -> list[float]:
    return [2.0 * xi * (y_hat - y) for xi in x_features]


def update_weight_vec(weights: list[float], dl_dweights: list[float], lr: float) -> list[float]:
    return [w - lr * dw for w, dw in zip(weights, dl_dweights, strict=True)]


def implement_linear_regression_vec(
    x_features: list[list[float]],
    y_output: list[float],
    epoch_max: int = 50,
    lr: float = 1e-5,
) -> tuple[list[float], list[float]]:
    """1-sample SGD using unified bias-augmented feature vectors [1, x1, x2, x3]."""
    losses: list[float] = []
    weights = initialize_params_vector()
    n = len(y_output)

    for _epoch in range(epoch_max):
        for i in range(n):
            features_i = x_features[i]
            y = y_output[i]

            y_hat = predict_vec(features_i, weights)
            loss = compute_loss_mse(y_hat, y)
            dl_dweights = compute_gradient_w(features_i, y, y_hat)
            weights = update_weight_vec(weights, dl_dweights, lr)
            losses.append(loss)

    return weights, losses


X_aug, y_aug = prepare_data_augmented(DATA_PATH)
weights_vec, losses_vec = implement_linear_regression_vec(X_aug, y_aug, epoch_max=50, lr=1e-5)

print("First sample X[0]:", X_aug[0])
print("Unified weights [b, w1, w2, w3]:", [round(w, 6) for w in weights_vec])

# Exact parity check against Problem 2 scalar SGD
assert np.allclose(weights_vec, [b_mse, w1_mse, w2_mse, w3_mse], atol=1e-12)
assert np.allclose(losses_vec, losses_sgd_mse, atol=1e-12)
print("Exact numerical parity confirmed between Problem 2 and Problem 4!")

plt.figure(figsize=(8, 4))
plt.plot(losses_vec[-100:], color="#0284c7")
plt.xlabel("#iteration (last 100)")
plt.ylabel("MSE loss")
plt.title("Bias-Augmented SGD: Last 100 Iterations")
plt.tight_layout()
plt.show()